In [0]:
%sql
CREATE CATALOG IF NOT EXISTS f1 MANAGED LOCATION 'abfss://jfcontainer@sauksdatabricksdata.dfs.core.windows.net/f1';
USE CATALOG f1;

CREATE SCHEMA IF NOT EXISTS source;
CREATE EXTERNAL VOLUME IF NOT EXISTS source.data LOCATION 'abfss://jfcontainer@sauksdatabricksdata.dfs.core.windows.net/f1/data';

CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

DROP TABLE IF EXISTS f1.bronze.driver_details;
DROP TABLE IF EXISTS f1.bronze.bahrain;
DROP TABLE IF EXISTS f1.gold.bahrain;

In [0]:
df = spark.read.format("json").load("/Volumes/f1/source/data/driver_details.json")
df.show()

df.write.option("overwriteSchema", "true").saveAsTable("`f1`.`bronze`.`driver_details`",mode="overwrite")

In [0]:
%sql
USE CATALOG f1;

CREATE TABLE IF NOT EXISTS bronze.bahrain
AS SELECT * FROM read_files(
  '/Volumes/f1/source/data/bahrain_2021.csv',
  format => 'csv',
  header => true
);

SELECT * FROM bronze.bahrain;

In [0]:
# Imaginary Silver Tables

In [0]:
%sql
USE CATALOG f1;

CREATE TABLE IF NOT EXISTS gold.bahrain AS SELECT driver_details.name, driver_details.team, AVG(to_timestamp(substring(bronze.bahrain.LapTime FROM 8), 'HH:mm:ss.SSSSSS')) as avg_lap_time
FROM bronze.driver_details
JOIN bronze.bahrain 
ON bronze.driver_details.driver_number = bronze.bahrain.DriverNumber
GROUP BY driver_details.name, driver_details.team;

SELECT * FROM gold.bahrain;